Honors Capstone - 2015 NFCS Analysis
Cleaned Code for Models 1, 2, and 3

In [2]:
# Necessary Imports

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2_contingency

In [5]:
# Survey Settings and Imports

In [6]:
CSV_PATH = "2015_Data.csv"
DK_CODES = [98, 99]
WEIGHT_COL = "WGT1"


HUMAN_STYLE_COL = "C1"   
ROBO_AUTO_COL = "C11"    


AGE_COL = "S_Age"
INCOME_COL = "S_Income"
EDU_COL = "S_Education"
ETH_COL = "S_Ethnicity"
GENDER_CANDIDATES = ["S_Gender2", "S_Gender"]

AGE_MAP = {
    1: "18-34",
    2: "35-54",
    3: "55+"
}

In [7]:
# Extra Helper Functions

In [8]:
def load_clean_data(path: str, dk_codes=None) -> pd.DataFrame:
    if dk_codes is None:
        dk_codes = DK_CODES
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    df.replace({c: np.nan for c in dk_codes}, inplace=True)
    return df


def to_numeric_nan(series: pd.Series, dk_codes=None) -> pd.Series:
    if dk_codes is None:
        dk_codes = DK_CODES
    s = pd.to_numeric(series, errors="coerce")
    return s.replace(dk_codes, np.nan)


def zscore(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    mu = s.mean(skipna=True)
    sd = s.std(skipna=True, ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.nan, index=s.index)
    return (s - mu) / sd

def weighted_mean(x: pd.Series, w: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce")
    w = pd.to_numeric(w, errors="coerce")
    mask = x.notna() & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float((x[mask] * w[mask]).sum() / w[mask].sum())


def require_columns(df: pd.DataFrame, cols: list[str]) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")


def pick_gender_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"No gender column found. Tried: {candidates}")

def classify_advisory_type(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    uses_human = out[HUMAN_STYLE_COL].isin([2, 3])
    uses_robo = out[ROBO_AUTO_COL] == 1

    out["advisory_type"] = "unclassified"
    out.loc[uses_human & (~uses_robo), "advisory_type"] = "traditional"
    out.loc[(~uses_human) & uses_robo, "advisory_type"] = "robo"
    out.loc[uses_human & uses_robo, "advisory_type"] = "hybrid"
    return out

def keep_classified(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["advisory_type"].isin(["traditional", "hybrid", "robo"])].copy()

def print_weighted_means(df: pd.DataFrame, columns: list[str], weight_col: str = WEIGHT_COL) -> None:
    if weight_col not in df.columns:
        print("\nNo weight column found; skipping weighted means.")
        return

    w = df[weight_col]
    print("\nWeighted means (overall, WGT1):")
    for col in columns:
        print(f"{col}: {weighted_mean(df[col], w):.4f}")


def print_weighted_means_by_group(df, group_col, value_col, groups=None, weight_col=WEIGHT_COL):
    if weight_col not in df.columns:
        print("\nNo weight column found; skipping weighted subgroup means.")
        return

    if groups is None:
        groups = sorted(df[group_col].dropna().unique())

    w = df[weight_col]
    print(f"\nWeighted {value_col} by {group_col}:")
    for grp in groups:
        mask = df[group_col] == grp
        print(f"{grp}: {weighted_mean(df.loc[mask, value_col], w.loc[mask]):.4f}")

In [14]:
# Load data

In [16]:
df = load_clean_data(CSV_PATH)
gender_col = pick_gender_col(df, GENDER_CANDIDATES)
df = classify_advisory_type(df)
df_classified = keep_classified(df)
df = df[df["advisory_type"].isin(["traditional", "hybrid", "robo"])].copy()

In [18]:
# MODEL 1: Demographic Predictors from Advisory Type

In [20]:
def subgroup_crosstab(df: pd.DataFrame, group_col: str, group_map: dict | None = None):
    tmp = df.copy()
    tmp["group_label"] = tmp[group_col].map(group_map) if group_map else tmp[group_col]
    tmp = tmp.dropna(subset=["group_label", "advisory_type"])

    counts = pd.crosstab(tmp["group_label"], tmp["advisory_type"])
    row_pct = counts.div(counts.sum(axis=1), axis=0) * 100

    if counts.shape[0] >= 2 and counts.shape[1] >= 2:
        chi2, p, dof, _ = chi2_contingency(counts)
    else:
        chi2, p, dof = np.nan, np.nan, np.nan

    stats = {"chi2": chi2, "p": p, "dof": dof}
    return counts, row_pct, stats

def build_model1_logit_data(df: pd.DataFrame, gender_col: str):
    use = df[[
        "advisory_type", AGE_COL, INCOME_COL, EDU_COL, gender_col, ETH_COL
    ]].dropna().copy()

    use["robo_binary"] = (use["advisory_type"] == "robo").astype(int)

    y = use["robo_binary"]
    X_raw = use[[AGE_COL, INCOME_COL, EDU_COL, gender_col, ETH_COL]].copy()

    for col in X_raw.columns:
        X_raw[col] = X_raw[col].astype("category")

    X = pd.get_dummies(X_raw, drop_first=True)
    X = sm.add_constant(X, has_constant="add").astype(float)

    return y, X, use

def odds_ratio_table(result) -> pd.DataFrame:
    conf = result.conf_int()
    out = pd.DataFrame({
        "term": result.params.index,
        "coef": result.params.values,
        "odds_ratio": np.exp(result.params.values),
        "ci_lower": np.exp(conf[0].values),
        "ci_upper": np.exp(conf[1].values),
        "p_value": result.pvalues.values
    })
    return out.sort_values("p_value")

dist_all = df["advisory_type"].value_counts(dropna=False)
dist_all_pct = (df["advisory_type"].value_counts(normalize=True, dropna=False) * 100).round(2)

age_counts, age_pct, age_stats = subgroup_crosstab(df_classified, AGE_COL, AGE_MAP)
inc_counts, inc_pct, inc_stats = subgroup_crosstab(df_classified, INCOME_COL)
edu_counts, edu_pct, edu_stats = subgroup_crosstab(df_classified, EDU_COL)

y, X, model1_df = build_model1_logit_data(df_classified, gender_col)
model1 = sm.Logit(y, X).fit(disp=False)

or_table = odds_ratio_table(model1)
marginal_effects = (
    model1.get_margeff(at="overall")
    .summary_frame()
    .reset_index()
    .rename(columns={"index": "term"})
)

In [22]:
print("\nMODEL 1 — DEMOGRAPHIC PREDICTORS OF ADVISORY TYPE")
print("\nGender column used:")
print(gender_col)

print("\n1) Advisory Type Distribution (All Cases)")
print(pd.DataFrame({"count": dist_all, "percent": dist_all_pct}))

print("\n2) Age vs Advisory Type (Row %)")
print(age_pct.round(2))
print(f"Chi-square = {age_stats['chi2']:.4f}, p = {age_stats['p']:.4f}, dof = {age_stats['dof']}")

print("\n3) Income vs Advisory Type (Row %)")
print(inc_pct.round(2))
print(f"Chi-square = {inc_stats['chi2']:.4f}, p = {inc_stats['p']:.4f}, dof = {inc_stats['dof']}")

print("\n4) Education vs Advisory Type (Row %)")
print(edu_pct.round(2))
print(f"Chi-square = {edu_stats['chi2']:.4f}, p = {edu_stats['p']:.4f}, dof = {edu_stats['dof']}")

print("\n5) Logistic Regression: Predicting Robo Use")
print(f"N used in regression: {len(model1_df)}")
print(f"Pseudo R-squared: {model1.prsquared:.4f}")

print("\nOdds Ratios")
print(or_table.round(4))

print("\nMarginal Effects")
print(marginal_effects.round(4))


MODEL 1 — DEMOGRAPHIC PREDICTORS OF ADVISORY TYPE

Gender column used:
S_Gender

1) Advisory Type Distribution (All Cases)
               count  percent
advisory_type                
traditional     1055    80.41
robo             154    11.74
hybrid           103     7.85

2) Age vs Advisory Type (Row %)
advisory_type  hybrid   robo  traditional
group_label                              
18-34           16.30  33.48        50.22
35-54            9.39  15.23        75.38
55+              4.20   2.60        93.20
Chi-square = 219.2194, p = 0.0000, dof = 4

3) Income vs Advisory Type (Row %)
advisory_type  hybrid   robo  traditional
group_label                              
1                9.80  14.51        75.69
2                8.39  13.26        78.36
3                6.07   8.24        85.68
Chi-square = 13.5634, p = 0.0088, dof = 4

4) Education vs Advisory Type (Row %)
advisory_type  hybrid   robo  traditional
group_label                              
1                8.30  13.56 

In [24]:
# MODEL 2: Planning Index

In [26]:
COMFORT_COL = "B10"         
KNOWLEDGE_COL = "C13"       
TRADE_FREQ_COL = "B3"
INFO_SOURCE_COLS = ["C11"]   

require_columns(df, [COMFORT_COL, KNOWLEDGE_COL, TRADE_FREQ_COL] + INFO_SOURCE_COLS)

model2_df = df_classified.copy()

model2_df["b3_trade_rev"] = 5 - to_numeric_nan(model2_df[TRADE_FREQ_COL])

info_used_cols = []
for col in INFO_SOURCE_COLS:
    yes_col = f"{col}_yes"
    model2_df[yes_col] = np.where(
        model2_df[col] == 1, 1,
        np.where(model2_df[col] == 2, 0, np.nan)
    )
    info_used_cols.append(yes_col)

model2_df["info_proxy_count"] = np.where(
    model2_df[info_used_cols].notna().sum(axis=1) > 0,
    model2_df[info_used_cols].sum(axis=1, skipna=True),
    np.nan
)

model2_df["z_comfort"] = zscore(to_numeric_nan(model2_df[COMFORT_COL]))
model2_df["z_knowledge"] = zscore(to_numeric_nan(model2_df[KNOWLEDGE_COL]))
model2_df["z_infoaccess_proxy"] = zscore(model2_df["info_proxy_count"])
model2_df["z_trade_freq"] = zscore(model2_df["b3_trade_rev"])

planning_components = [
    "z_comfort",
    "z_knowledge",
    "z_infoaccess_proxy",
    "z_trade_freq"
]

model2_df["planning_index"] = np.where(
    model2_df[planning_components].notna().all(axis=1),
    model2_df[planning_components].mean(axis=1),
    np.nan
)

# Summary stats
n = int(model2_df["planning_index"].notna().sum())
mean_unw = float(model2_df["planning_index"].mean(skipna=True))

In [28]:
print("\nMODEL 2 — PREPARDNESS/PLANNING INDEX (2015)")
print("-----------------------------------")
print(f"Non-missing N: {n}")
print(f"Unweighted overall mean: {mean_unw:.4f}")

if WEIGHT_COL in model2_df.columns:
    print(f"Weighted overall mean (WGT1): {weighted_mean(model2_df['planning_index'], model2_df[WEIGHT_COL]):.4f}")
else:
    print("No weight column found; skipping weighted mean.")

print("\nUnweighted mean by advisory type:")
print(
    model2_df.groupby("advisory_type")["planning_index"]
    .mean()
    .reindex(["traditional", "hybrid", "robo"])
    .round(4)
)

print("\nWeighted mean by advisory type:")
for grp in ["traditional", "hybrid", "robo"]:
    sub = model2_df[model2_df["advisory_type"] == grp]
    print(f"{grp}: {weighted_mean(sub['planning_index'], sub[WEIGHT_COL]):.4f}")

print("\nComponents used:")
print(planning_components)


MODEL 2 — PREPARDNESS/PLANNING INDEX (2015)
-----------------------------------
Non-missing N: 1160
Unweighted overall mean: 0.0003
Weighted overall mean (WGT1): 0.0185

Unweighted mean by advisory type:
advisory_type
traditional   -0.0682
hybrid         0.3579
robo           0.1811
Name: planning_index, dtype: float64

Weighted mean by advisory type:
traditional: -0.0575
hybrid: 0.3296
robo: 0.1782

Components used:
['z_comfort', 'z_knowledge', 'z_infoaccess_proxy', 'z_trade_freq']


In [30]:
# MODEL 3: Bounded Rationality Index

In [32]:
model3_df = df_classified.copy()


LITERACY_KEY = {
    "G4": 1,
    "G5": 2,
    "G6": 2,
    "G7": 1,
    "G8": 1,
}

lit_items = [c for c in LITERACY_KEY if c in model3_df.columns]
if len(lit_items) == 0:
    raise ValueError("None of the expected 2015 literacy columns were found.")

lit_correct = []
for col in lit_items:
    s = to_numeric_nan(model3_df[col])
    lit_correct.append((s == LITERACY_KEY[col]).astype(float))

lit_df = pd.concat(lit_correct, axis=1)
model3_df["literacy_score"] = lit_df.sum(axis=1, min_count=1)
model3_df["literacy_pct"] = lit_df.mean(axis=1)

HEURISTIC_COLS = ["F1_5", "F1_9", "F2_6"]
heuristic_items = [c for c in HEURISTIC_COLS if c in model3_df.columns]

if len(heuristic_items) == 0:
    model3_df["heuristic_reliance"] = np.nan
else:
    converted = []
    for c in heuristic_items:
        s = to_numeric_nan(model3_df[c])
        converted.append(np.where(s == 1, 1, np.where(s == 2, 0, np.nan)))
    model3_df["heuristic_reliance"] = pd.DataFrame(converted).T.mean(axis=1)

SELF_RATED_KNOW_COL = "G2"
if SELF_RATED_KNOW_COL not in model3_df.columns:
    raise ValueError("Missing G2 (self-rated investment knowledge).")

model3_df["self_rated_knowledge"] = to_numeric_nan(model3_df[SELF_RATED_KNOW_COL])

model3_df["overconfidence_gap"] = (
    zscore(model3_df["self_rated_knowledge"])
    - zscore(model3_df["literacy_score"])
)

model3_df["BRI"] = np.where(
    model3_df[["literacy_score", "heuristic_reliance", "overconfidence_gap"]].notna().all(axis=1),
    (
        (-1 * zscore(model3_df["literacy_score"])) +
        zscore(model3_df["heuristic_reliance"]) +
        zscore(model3_df["overconfidence_gap"])
    ) / 3,
    np.nan
)

In [34]:
print("\nMODEL 3 — BOUNDED RATIONALITY INDEX (2015)")
print("------------------------------------------------")

print("\nAdvisory type counts:")
print(model3_df["advisory_type"].value_counts().reindex(["traditional", "hybrid", "robo"]))

print("\nNon-missing counts:")
print(
    model3_df[[
        "literacy_score",
        "heuristic_reliance",
        "overconfidence_gap",
        "BRI"
    ]].notna().sum()
)

print("\nUnweighted means by advisory type:")
print(
    model3_df.groupby("advisory_type")[[
        "literacy_score",
        "heuristic_reliance",
        "overconfidence_gap",
        "BRI"
    ]]
    .mean(numeric_only=True)
    .reindex(["traditional", "hybrid", "robo"])
    .round(4)
)

print("\nWeighted means (overall, WGT1):")
for col in ["literacy_score", "heuristic_reliance", "overconfidence_gap", "BRI"]:
    print(f"{col}: {weighted_mean(model3_df[col], model3_df[WEIGHT_COL]):.4f}")

print("\nWeighted means by advisory type:")
for grp in ["traditional", "hybrid", "robo"]:
    sub = model3_df[model3_df["advisory_type"] == grp]
    print(f"\n{grp}:")
    for col in ["literacy_score", "heuristic_reliance", "overconfidence_gap", "BRI"]:
        print(f"  {col}: {weighted_mean(sub[col], sub[WEIGHT_COL]):.4f}")


MODEL 3 — BOUNDED RATIONALITY INDEX (2015)
------------------------------------------------

Advisory type counts:
advisory_type
traditional    1055
hybrid          103
robo            154
Name: count, dtype: int64

Non-missing counts:
literacy_score        1312
heuristic_reliance     857
overconfidence_gap    1306
BRI                    852
dtype: int64

Unweighted means by advisory type:
               literacy_score  heuristic_reliance  overconfidence_gap     BRI
advisory_type                                                                
traditional            3.3498              0.4562             -0.2642 -0.0772
hybrid                 3.0583              0.4437              0.7439  0.2665
robo                   2.9675              0.4603              1.2416  0.4060

Weighted means (overall, WGT1):
literacy_score: 3.1637
heuristic_reliance: 0.4648
overconfidence_gap: 0.1301
BRI: 0.0880

Weighted means by advisory type:

traditional:
  literacy_score: 3.2337
  heuristic_reliance: